## Импорты

In [ ]:
try:
    import google.colab
    %pip install pandas torch transformers datasets evaluate sacrebleu bert-score sentence-transformers rapidfuzz deep-translator tqdm numpy matplotlib PyYAML
    %pip install --upgrade tiktoken protobuf sentencepiece
    %pip install accelerate>=1.1.0
    is_colab = True
except:
    is_colab = False

if is_colab:
    !git clone https://github.com/AbonentVneSeti/text_processing_final_project
    %cd text_processing_final_project

In [ ]:
import sys
sys.path.append('..')
from utils import load_config, build_dataloaders
import pandas as pd
from models.model_use import load_model, evaluate_model

config = load_config("config.yaml")


## Загрузка данных и модели

In [ ]:
df = pd.read_parquet("data/paraphrases_clean.parquet")
_, val_loader = build_dataloaders(df, config["model_config"], split_ratios=(0.9, 0.1))

In [ ]:
model = load_model(
    config["model_name"],
    config["model_config"],
    checkpoint_path=config["trainer_config"]["output_dir"]
)

## Тестовые метрики

In [ ]:
metrics = evaluate_model(model, val_loader, config["metrics_config"])
print("Метрики на тестовой выборке:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

## Проверка на готовых предложениях

In [ ]:
test_sentences = [
    "сегодня хорошая погода.",
    "искусственный интеллект меняет мир.",
    "я люблю читать книги по вечерам.",
    "машинное обучение требует больших данных.",
    "кошка сидит на подоконнике и смотрит в окно.",
    "перефразирование помогает улучшить текст.",
    "экономика страны растёт благодаря новым технологиям.",
    "студенты сдают экзамены в конце семестра."
]

paraphrases = model.generate(test_sentences)

print("Ручная проверка:\n")
for orig, para in zip(test_sentences, paraphrases):
    print(f"Оригинал : {orig}")
    print(f"Парафраз : {para}")
    print("-" * 50)